# Data Governance – Medallion Pipeline

**Author:** Jarosław Błaziński  
**Project:** E-Commerce Medallion Architecture (Databricks Community Edition)  
**Stack:** PySpark, Delta Tables, Databricks  

---

## Co zawiera ten notebook?

Data Governance to zbiór praktyk zapewniających że dane są:
- **wiarygodne** – wiemy skąd pochodzą i czy są poprawne
- **bezpieczne** – dane wrażliwe są wykryte i chronione
- **udokumentowane** – każdy w zespole wie co jest w każdej tabeli

Ten notebook implementuje cztery filary governance dla naszego pipeline'u:

| Sekcja | Co robi |
|---|---|
| 1. Data Lineage | Dokumentacja przepływu Bronze → Silver → Gold |
| 2. PII Detection | Wykrywanie kolumn z danymi osobowymi |
| 3. Data Quality Checks | Formalne testy jakości danych w stylu Great Expectations |
| 4. Data Catalog | Automatyczna dokumentacja tabel i schematów |

---
# SEKCJA 1 – Data Lineage

Data Lineage odpowiada na pytanie: **skąd pochodzi każda kolumna i co się z nią działo?**  
Bez lineage nie możemy debugować błędów ani spełniać wymogów regulacyjnych (RODO, audyty).

In [ ]:
import logging
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as spark_sum, avg, min as spark_min, max as spark_max

spark = SparkSession.builder.appName("Data_Governance").getOrCreate()
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("GOVERNANCE")

BRONZE_PATH = "/tmp/medallion/bronze/transactions"
SILVER_PATH = "/tmp/medallion/silver/transactions"
GOLD_PATH   = "/tmp/medallion/gold"

logger.info("Data Governance notebook uruchomiony: %s", datetime.now())

In [ ]:
# Definicja lineage – skąd pochodzi każda kolumna i jakie transformacje przeszła

LINEAGE = {
    "pipeline": "E-Commerce ELT Medallion Pipeline",
    "owner":    "Jarosław Błaziński",
    "source":   "input_data_v2.csv (Frankfurter-style e-commerce transactions)",
    "layers": [
        {
            "layer": "Bronze",
            "path": BRONZE_PATH,
            "description": "Raw ingestion – load as-is, no transformations",
            "columns_added": ["_ingested_at", "_source_file"],
            "columns_removed": [],
            "transformations": ["None – raw data only"],
            "partitioned_by": ["country"]
        },
        {
            "layer": "Silver",
            "path": SILVER_PATH,
            "description": "Cleaned and enriched data – ready for analysis",
            "columns_added": [
                "country_name   (joined from country_dim)",
                "vat_rate       (joined from country_dim: PL=0.23, DE=0.19, DK/SE=0.25)",
                "amount_vat     (amount * (1 + vat_rate))",
                "revenue        (amount * quantity)",
                "revenue_vat    (amount_vat * quantity)",
                "transaction_date_only (to_date of transaction_date)",
                "month          (month(transaction_date))",
                "year           (year(transaction_date))",
                "day_of_week    (dayofweek(transaction_date))",
                "is_weekend     (day_of_week IN (1, 7))",
                "_processed_at  (current_timestamp)"
            ],
            "columns_removed": ["_ingested_at", "_source_file"],
            "transformations": [
                "FILTER: amount IS NOT NULL",
                "FILTER: amount > 0 (removes refunds)",
                "FILTER: status != 'cancelled'",
                "LEFT JOIN country_dim ON country (broadcast join)"
            ],
            "partitioned_by": ["country", "year"]
        },
        {
            "layer": "Gold",
            "path": GOLD_PATH,
            "description": "Aggregated tables ready for reporting and Power BI",
            "tables": [
                "revenue_by_country_category – GROUP BY country, category + dense_rank()",
                "monthly_trend              – GROUP BY year, month, country",
                "payment_methods            – GROUP BY country, payment_method"
            ],
            "transformations": [
                "GROUP BY + SUM/AVG/COUNT aggregations",
                "WINDOW FUNCTION: dense_rank() OVER (PARTITION BY country ORDER BY total_revenue DESC)",
                "MERGE INTO (upsert pattern for incremental loads)"
            ],
            "partitioned_by": ["N/A – Gold tables are small enough for single partition"]
        }
    ]
}

# Wypisz lineage w czytelnej formie
import json
print("=" * 60)
print("DATA LINEAGE REPORT")
print("=" * 60)
print(f"Pipeline : {LINEAGE['pipeline']}")
print(f"Owner    : {LINEAGE['owner']}")
print(f"Source   : {LINEAGE['source']}")
print()

for layer in LINEAGE["layers"]:
    print(f">>> Layer: {layer['layer']}")
    print(f"    Path : {layer['path']}")
    print(f"    Desc : {layer['description']}")
    if "transformations" in layer:
        print(f"    Transformations:")
        for t in layer["transformations"]:
            print(f"      - {t}")
    print()

In [ ]:
# Sprawdź ile wierszy zostało w każdej warstwie – potwierdzenie lineage

from delta.tables import DeltaTable

df_bronze = spark.read.format("delta").load(BRONZE_PATH)
df_silver = spark.read.format("delta").load(SILVER_PATH)
df_gold   = spark.read.format("delta").load(f"{GOLD_PATH}/revenue_by_country_category")

bronze_count = df_bronze.count()
silver_count = df_silver.count()
gold_count   = df_gold.count()

print("=" * 50)
print("ROW COUNT PER LAYER")
print("=" * 50)
print(f"Bronze : {bronze_count:>10,}  (100.0%)")
print(f"Silver : {silver_count:>10,}  ({silver_count/bronze_count*100:.1f}% retained after cleaning)")
print(f"Gold   : {gold_count:>10,}  (aggregated rows – country x category combinations)")

---
# SEKCJA 2 – PII Detection

PII (Personally Identifiable Information) to dane które mogą zidentyfikować osobę fizyczną.  
RODO wymaga żebyśmy wiedzieli gdzie przechowujemy PII i odpowiednio je chronili.  

**Strategie ochrony PII:**
- **Pseudonymizacja** – zastąpienie ID hashem (customer_id → hash)
- **Maskowanie** – ukrycie części danych (jan.kowalski@email.com → j***@email.com)
- **Usunięcie** – jeśli kolumna nie jest potrzebna w danej warstwie

In [ ]:
from pyspark.sql.functions import sha2, concat_ws, lit, regexp_replace, when

# Definicja kolumn PII w naszym datasecie
PII_REGISTRY = {
    "customer_id": {
        "pii_type": "IDENTIFIER",
        "risk": "HIGH",
        "description": "Unikalny identyfikator klienta – może być powiązany z danymi osobowymi w systemie CRM",
        "action": "Pseudonymizacja (SHA-256 hash) w warstwie Silver"
    },
    "transaction_id": {
        "pii_type": "IDENTIFIER",
        "risk": "MEDIUM",
        "description": "ID transakcji – nie zawiera danych osobowych ale może linkować do danych PII",
        "action": "Zachować – potrzebny do audytu i debugowania"
    },
    "amount": {
        "pii_type": "FINANCIAL",
        "risk": "MEDIUM",
        "description": "Kwota transakcji – dane finansowe wrażliwe w kontekście konkretnej osoby",
        "action": "Zachować – niezbędne do analizy, agregacje w Gold nie ujawniają danych jednostkowych"
    }
}

print("=" * 60)
print("PII REGISTRY – WYKAZ KOLUMN Z DANYMI WRAŻLIWYMI")
print("=" * 60)
for col_name, info in PII_REGISTRY.items():
    print(f"\nKolumna  : {col_name}")
    print(f"  Typ PII  : {info['pii_type']}")
    print(f"  Ryzyko   : {info['risk']}")
    print(f"  Opis     : {info['description']}")
    print(f"  Akcja    : {info['action']}")

In [ ]:
# Pseudonymizacja customer_id – zamieniamy ID na hash SHA-256
# Dzięki temu Gold nie zawiera bezpośrednich identyfikatorów klientów

df_pseudonymized = (
    df_silver
    .withColumn(
        "customer_id_hash",
        sha2(concat_ws("-", col("customer_id"), lit("SALT_2024")), 256)
    )
    .drop("customer_id")   # usuwamy oryginalny ID z widoku analitycznego
)

print("Przed pseudonymizacją – customer_id widoczny:")
df_silver.select("transaction_id", "customer_id", "amount", "country").show(5)

print("Po pseudonymizacji – customer_id zastąpiony hashem:")
df_pseudonymized.select("transaction_id", "customer_id_hash", "amount", "country").show(5, truncate=True)

In [ ]:
# Automatyczne wykrywanie potencjalnych kolumn PII po nazwie
# W produkcji użylibyśmy narzędzia jak AWS Macie lub Azure Purview
# Tutaj implementujemy prosty skaner nazw kolumn

PII_KEYWORDS = [
    "name", "email", "phone", "address", "ip", "ssn",
    "password", "card", "iban", "pesel", "nip", "id",
    "customer", "user", "birth", "age", "gender"
]

def scan_for_pii(df, layer_name):
    print(f"\n=== PII SCAN – warstwa {layer_name} ===")
    flagged = []
    for col_name in df.columns:
        for keyword in PII_KEYWORDS:
            if keyword.lower() in col_name.lower():
                flagged.append((col_name, keyword))
                break
    if flagged:
        print(f"Potencjalne kolumny PII ({len(flagged)}):")
        for col_name, keyword in flagged:
            print(f"  ⚠️  {col_name:<30} (słowo kluczowe: '{keyword}')")
    else:
        print("Brak wykrytych kolumn PII.")
    return flagged

scan_for_pii(df_bronze, "Bronze")
scan_for_pii(df_silver, "Silver")
scan_for_pii(df_gold,   "Gold")

---
# SEKCJA 3 – Data Quality Checks

Formalne testy jakości danych – wzorowany na Great Expectations.  
Każdy check zwraca PASS/FAIL z detalami. Przy FAIL pipeline powinien się zatrzymać.  

**Rodzaje testów:**
- **Completeness** – czy nie brakuje danych (NULLe)
- **Validity** – czy wartości są w dozwolonym zakresie
- **Uniqueness** – czy klucze główne są unikalne
- **Consistency** – czy dane są spójne między warstwami
- **Freshness** – czy dane są aktualne

In [ ]:
from pyspark.sql.functions import countDistinct, isnull

class DataQualityCheck:
    """Prosty framework do testowania jakości danych w stylu Great Expectations."""

    def __init__(self, df, layer_name):
        self.df         = df
        self.layer_name = layer_name
        self.results    = []

    def _record(self, check_name, check_type, passed, details):
        status = "PASS" if passed else "FAIL"
        self.results.append({
            "check": check_name,
            "type":  check_type,
            "status": status,
            "details": details
        })
        icon = "✅" if passed else "❌"
        print(f"  {icon} [{check_type}] {check_name}: {status} – {details}")

    def expect_no_nulls(self, column):
        null_count = self.df.filter(col(column).isNull()).count()
        passed = null_count == 0
        self._record(
            f"expect_no_nulls('{column}')",
            "COMPLETENESS",
            passed,
            f"{null_count} NULLs found"
        )
        return self

    def expect_null_rate_below(self, column, threshold):
        total = self.df.count()
        null_count = self.df.filter(col(column).isNull()).count()
        rate = null_count / total if total > 0 else 0
        passed = rate <= threshold
        self._record(
            f"expect_null_rate_below('{column}', {threshold:.0%})",
            "COMPLETENESS",
            passed,
            f"actual rate: {rate:.1%} ({null_count:,} NULLs / {total:,} rows)"
        )
        return self

    def expect_values_in_set(self, column, valid_values):
        invalid_count = self.df.filter(~col(column).isin(valid_values)).count()
        passed = invalid_count == 0
        self._record(
            f"expect_values_in_set('{column}')",
            "VALIDITY",
            passed,
            f"{invalid_count} rows with values outside {valid_values}"
        )
        return self

    def expect_column_values_positive(self, column):
        non_positive = self.df.filter(col(column) <= 0).count()
        passed = non_positive == 0
        self._record(
            f"expect_column_values_positive('{column}')",
            "VALIDITY",
            passed,
            f"{non_positive} rows with value <= 0"
        )
        return self

    def expect_column_values_between(self, column, min_val, max_val):
        out_of_range = self.df.filter(
            (col(column) < min_val) | (col(column) > max_val)
        ).count()
        passed = out_of_range == 0
        self._record(
            f"expect_column_values_between('{column}', {min_val}, {max_val})",
            "VALIDITY",
            passed,
            f"{out_of_range} rows outside range [{min_val}, {max_val}]"
        )
        return self

    def expect_unique(self, column):
        total  = self.df.count()
        unique = self.df.select(countDistinct(column)).collect()[0][0]
        passed = total == unique
        self._record(
            f"expect_unique('{column}')",
            "UNIQUENESS",
            passed,
            f"{total - unique} duplicates found ({unique:,} unique / {total:,} total)"
        )
        return self

    def expect_row_count_between(self, min_rows, max_rows):
        total = self.df.count()
        passed = min_rows <= total <= max_rows
        self._record(
            f"expect_row_count_between({min_rows:,}, {max_rows:,})",
            "COMPLETENESS",
            passed,
            f"actual: {total:,} rows"
        )
        return self

    def summary(self):
        total  = len(self.results)
        passed = sum(1 for r in self.results if r["status"] == "PASS")
        failed = total - passed
        print(f"\n{'='*50}")
        print(f"QUALITY SUMMARY – {self.layer_name}")
        print(f"  Total : {total}  |  Passed : {passed}  |  Failed : {failed}")
        print(f"  Score : {passed/total*100:.0f}%")
        if failed > 0:
            print("  ⚠️  PIPELINE SHOULD STOP – fix issues before proceeding")
        else:
            print("  ✅  All checks passed – safe to proceed")
        print(f"{'='*50}")
        return failed == 0

print("DataQualityCheck class zdefiniowana.")

In [ ]:
# Testy jakości dla warstwy Bronze
print("\n>>> BRONZE QUALITY CHECKS\n")

bronze_qc = DataQualityCheck(df_bronze, "Bronze")
bronze_ok = (
    bronze_qc
    .expect_row_count_between(400_000, 600_000)
    .expect_no_nulls("transaction_id")
    .expect_unique("transaction_id")
    .expect_null_rate_below("amount", threshold=0.15)   # akceptujemy do 15% NULLi w Bronze
    .expect_values_in_set("country", ["PL", "DE", "DK", "SE"])
    .expect_values_in_set("status", ["completed", "refunded", "cancelled"])
    .summary()
)

In [ ]:
# Testy jakości dla warstwy Silver – wyższe wymagania niż Bronze
print("\n>>> SILVER QUALITY CHECKS\n")

silver_qc = DataQualityCheck(df_silver, "Silver")
silver_ok = (
    silver_qc
    .expect_row_count_between(300_000, 500_000)
    .expect_no_nulls("transaction_id")
    .expect_no_nulls("amount")              # Silver nie może mieć NULLi w amount
    .expect_no_nulls("country_name")        # join z country_dim musi dać wynik
    .expect_column_values_positive("amount")   # Silver nie może mieć ujemnych
    .expect_column_values_positive("revenue")
    .expect_column_values_between("vat_rate", 0.10, 0.30)
    .expect_column_values_between("month", 1, 12)
    .expect_values_in_set("status", ["completed", "refunded"])  # cancelled usunięte
    .summary()
)

In [ ]:
# Testy spójności między warstwami (Cross-layer consistency)
print("\n>>> CROSS-LAYER CONSISTENCY CHECKS\n")

bronze_count = df_bronze.count()
silver_count = df_silver.count()

retention_rate = silver_count / bronze_count

# Oczekujemy że Silver ma 65-90% wierszy Bronze (reszta to NULLe, zwroty, anulowane)
retention_ok = 0.65 <= retention_rate <= 0.90

print(f"  {'✅' if retention_ok else '❌'} [CONSISTENCY] retention_rate: "
      f"{'PASS' if retention_ok else 'FAIL'} – "
      f"{retention_rate:.1%} ({silver_count:,} / {bronze_count:,})")

# Sprawdź czy Gold nie ma więcej wierszy niż możliwych kombinacji country x category
gold_count = df_gold.count()
max_combinations = 4 * 8  # 4 kraje x 8 kategorii
gold_ok = gold_count <= max_combinations

print(f"  {'✅' if gold_ok else '❌'} [CONSISTENCY] gold_row_count: "
      f"{'PASS' if gold_ok else 'FAIL'} – "
      f"{gold_count} rows (max expected: {max_combinations})")

---
# SEKCJA 4 – Data Catalog

Data Catalog to dokumentacja wszystkich tabel: co zawierają, kto jest właścicielem, kiedy były ostatnio aktualizowane.  
W produkcji używa się narzędzi jak Azure Purview, AWS Glue Data Catalog, Databricks Unity Catalog.  
Tutaj generujemy go automatycznie z Delta metadata.

In [ ]:
from pyspark.sql.functions import current_timestamp

def generate_catalog_entry(path, layer, description, owner, tags):
    """Generuje wpis do data catalog dla Delta Table."""
    df = spark.read.format("delta").load(path)
    delta_table = DeltaTable.forPath(spark, path)
    history = delta_table.history(1).collect()[0]

    print(f"\n{'='*60}")
    print(f"TABLE: {layer}")
    print(f"{'='*60}")
    print(f"Path        : {path}")
    print(f"Layer       : {layer}")
    print(f"Description : {description}")
    print(f"Owner       : {owner}")
    print(f"Tags        : {', '.join(tags)}")
    print(f"Row Count   : {df.count():,}")
    print(f"Columns     : {len(df.columns)}")
    print(f"Last Updated: {history['timestamp']}")
    print(f"Version     : {history['version']}")
    print(f"Last Op     : {history['operation']}")
    print()
    print("Schema:")
    for field in df.schema.fields:
        nullable = "nullable" if field.nullable else "NOT NULL"
        pii_flag = " ⚠️ PII" if any(kw in field.name.lower() for kw in ["customer", "id", "amount"]) else ""
        print(f"  {field.name:<30} {str(field.dataType):<20} {nullable}{pii_flag}")

    return df

generate_catalog_entry(
    path=BRONZE_PATH,
    layer="bronze.transactions",
    description="Raw e-commerce transactions loaded from CSV. No transformations applied.",
    owner="Jarosław Błaziński",
    tags=["bronze", "raw", "ecommerce", "transactions", "pii"]
)

In [ ]:
generate_catalog_entry(
    path=SILVER_PATH,
    layer="silver.transactions",
    description="Cleaned and enriched transactions. NULLs removed, VAT calculated, country dimension joined.",
    owner="Jarosław Błaziński",
    tags=["silver", "clean", "ecommerce", "transactions", "vat", "pii"]
)

In [ ]:
generate_catalog_entry(
    path=f"{GOLD_PATH}/revenue_by_country_category",
    layer="gold.revenue_by_country_category",
    description="Aggregated revenue per country and product category with ranking. Source for Power BI.",
    owner="Jarosław Błaziński",
    tags=["gold", "aggregated", "reporting", "powerbi", "no-pii"]
)

## Podsumowanie – Data Governance w projekcie

| Filar | Implementacja | Status |
|---|---|---|
| Data Lineage | Dokumentacja przepływu Bronze→Silver→Gold, count per layer | ✅ |
| PII Detection | Registry kolumn PII, automatyczny skaner, pseudonymizacja customer_id | ✅ |
| Data Quality | Framework testów: completeness, validity, uniqueness, consistency | ✅ |
| Data Catalog | Automatyczny katalog z Delta metadata: schemat, właściciel, history | ✅ |

**W produkcji te elementy byłyby rozszerzone o:**
- Unity Catalog (Databricks) – centralny katalog z kontrolą dostępu
- Azure Purview / AWS Macie – automatyczne skanowanie PII
- Great Expectations – zaawansowane testy z raportowaniem HTML
- Alerty na Slack/email gdy Quality Check = FAIL